In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from util.db_helpers import get_mysql_engine

#---------------------------------------------
# Config
#---------------------------------------------

# Set to True to force re-downloading tables from MySQL even if a cached
# parquet file already exists in output/pipeline.
FORCE_MYSQL_REFRESH = False
MYSQL_DB = '2023_parcel_baseyear_luvit_update_scenario'

In [2]:
# download tables from MySQL and cache them as parquet files in output/pipeline
output_dir = Path('output/pipeline')
output_dir.mkdir(parents=True, exist_ok=True)

table_names = ['parcels', 'buildings', 'growth_centers','households','jobs','building_types']
dataframes = {}
engine = None

for table in table_names:
    parquet_path = output_dir / f'{table}.parquet'
    if parquet_path.exists() and not FORCE_MYSQL_REFRESH:
        dataframes[table] = pd.read_parquet(parquet_path)
    else:
        if engine is None:
            engine = get_mysql_engine(MYSQL_DB)
        df = pd.read_sql_table(table, engine)
        df.to_parquet(parquet_path)
        dataframes[table] = df

parcels_df = dataframes['parcels']
buildings_df = dataframes['buildings']
growth_centers_df = dataframes['growth_centers']
households_df = dataframes['households']
jobs_df = dataframes['jobs']
building_types_df = dataframes['building_types']

In [3]:
# add is_residential and growth_center_id columns to buildings_df
is_residential = building_types_df[['building_type_id','is_residential']].set_index('building_type_id')['is_residential']
buildings_df['is_residential'] = buildings_df['building_type_id'].map(is_residential)
parcel_center_xwalk = parcels_df[['parcel_id','growth_center_id']].set_index('parcel_id')['growth_center_id']
buildings_df['growth_center_id'] = buildings_df['parcel_id'].map(parcel_center_xwalk)
county_xwalk = parcels_df[['parcel_id','county_id']].set_index('parcel_id')['county_id']
buildings_df['county_id'] = buildings_df['parcel_id'].map(county_xwalk)

In [4]:
far = buildings_df.loc[buildings_df['growth_center_id'] != 0].groupby(['county_id','growth_center_id','is_residential'])[['gross_sqft','land_area']].sum().reset_index()
far['is_residential'] = far['is_residential'].map({1: 'res', 0: 'non_res'})
far = far.pivot_table(index=['county_id','growth_center_id'], columns='is_residential', values=['gross_sqft','land_area'], aggfunc='sum')
far.columns = [f'{col}_{suffix}' for col, suffix in far.columns]
far = far.reset_index()

In [5]:
far['gross_sqft_all'] = far['gross_sqft_res'] + far['gross_sqft_non_res']
far['land_area_all'] = far['land_area_res'] + far['land_area_non_res']
for type in ['res','non_res','all']:
    far[f'far_{type}'] = far[f'gross_sqft_{type}'] / far[f'land_area_{type}']

In [6]:
building_parcel_xwalk = buildings_df[['building_id','parcel_id']].set_index('building_id')['parcel_id']
households_df['parcel_id'] = households_df['building_id'].map(building_parcel_xwalk)
households_df['growth_center_id'] = households_df['parcel_id'].map(parcel_center_xwalk)
households_df['county_id'] = households_df['parcel_id'].map(county_xwalk)
persons = households_df.loc[households_df['growth_center_id'] != 0].groupby(['county_id','growth_center_id'])[['persons']].sum().reset_index()

In [7]:
jobs_df['parcel_id'] = jobs_df['building_id'].map(building_parcel_xwalk)
jobs_df['growth_center_id'] = jobs_df['parcel_id'].map(parcel_center_xwalk)
jobs_df['county_id'] = jobs_df['parcel_id'].map(county_xwalk)
jobs_mask = (jobs_df['growth_center_id'] != 0) & (jobs_df['home_based_status'] == 0)
jobs = jobs_df.loc[jobs_mask].groupby(['county_id','growth_center_id']).size().reset_index(name='jobs')

In [8]:
au = persons.merge(jobs, on=['county_id','growth_center_id'], how='outer').merge(far, on=['county_id','growth_center_id'], how='outer')
au['activity_units'] = au['persons'] + au['jobs']
au['acres_land'] = au['land_area_all'] / 43560
au['au_acre'] = au['activity_units'] / au['acres_land']

In [9]:
county_map = {
    33: 'King',
    35: 'Kitsap',
    53: 'Pierce',
    61: 'Snohomish'
}
au['county'] = au['county_id'].map(county_map)

growth_centers = growth_centers_df[['growth_center_id','name']].set_index('growth_center_id')['name']
au['growth_center'] = au['growth_center_id'].map(growth_centers)

# Chart

In [10]:
#| title: "Activity Units per Acre vs. FAR"
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

valid = au[['county', 'growth_center', 'far_all', 'au_acre']].dropna()

slope, intercept = np.polyfit(valid['far_all'], valid['au_acre'], 1)
x_line = np.linspace(valid['far_all'].min(), valid['far_all'].max(), 100)

fig = px.scatter(
    valid,
    x='far_all',
    y='au_acre',
    hover_name='growth_center',
    hover_data={'county': True, 'far_all': ':.2f', 'au_acre': ':.1f'},
    labels={'far_all': 'FAR', 'au_acre': 'Activity Units / Acre'},
)
# Assign (rather than call as a bare statement) so these chainable methods'
# return values aren't treated as separate top-level expression outputs.
# Quarto dashboards set Jupyter's shell interactivity to "all" so every
# top-level expression is displayed - since Plotly's add_trace/update_layout
# return `self`, calling them as bare statements was producing extra charts.
fig = fig.add_trace(go.Scatter(
    x=x_line,
    y=slope * x_line + intercept,
    mode='lines',
    name=f'y = {slope:.2f}x + {intercept:.2f}',
    line=dict(color='red'),
))
fig = fig.update_layout(
    xaxis_title='FAR',
    yaxis_title='AU / Acre',
    legend_title_text='',
)
fig

# FAR Table

In [11]:
#| title: "FAR by Growth Center"
au[['county','growth_center','gross_sqft_non_res', 'gross_sqft_res', 'land_area_non_res',
       'land_area_res', 'gross_sqft_all', 'land_area_all', 'far_res',
       'far_non_res', 'far_all']]

,county,growth_center,gross_sqft_non_res,gross_sqft_res,land_area_non_res,land_area_res,gross_sqft_all,land_area_all,far_res,far_non_res,far_all
0,King,Auburn,3326850.0,1768626.0,2076877.0,849680.0,5095476.0,2926557.0,2.081520,1.601852,1.741116
1,King,Bellevue,40617329.0,10636173.0,5655822.0,1284029.0,51253502.0,6939851.0,8.283437,7.181508,7.385389
2,King,Burien,3381113.0,1806127.0,2465222.0,785208.0,5187240.0,3250430.0,2.300189,1.371525,1.595863
3,King,Federal Way,2494599.0,261635.0,1566212.0,37376.0,2756234.0,1603588.0,7.000080,1.592759,1.718792
4,King,Kent,2994430.0,1186542.0,1819170.0,303968.0,4180972.0,2123138.0,3.903510,1.646042,1.969242
5,King,Kirkland Totem Lake,7743882.0,5871758.0,4213091.0,1033508.0,13615640.0,5246599.0,5.681386,1.838052,2.595136
6,King,Redmond Downtown,5805982.0,8424325.0,2743656.0,1640602.0,14230307.0,4384258.0,5.134899,2.116148,3.245773
7,King,Redmond-Overlake,24173230.0,5922963.0,7751590.0,1143342.0,30096193.0,8894932.0,5.180395,3.118487,3.383521
8,King,Renton,10509394.0,3814252.0,4357776.0,1105974.0,14323646.0,5463750.0,3.448772,2.411642,2.621578
9,King,SeaTac,9012023.0,4138448.0,2686966.0,1653087.0,13150471.0,4340053.0,2.503467,3.353977,3.030025


# Activity Units Table

In [12]:
#| title: "Activity Units by Growth Center"
au[['county','growth_center','jobs','persons','activity_units','acres_land','au_acre']]

,county,growth_center,jobs,persons,activity_units,acres_land,au_acre
0,King,Auburn,5354,3612.0,8966.0,67.184504,133.453393
1,King,Bellevue,52050,16544.0,68594.0,159.317057,430.550258
2,King,Burien,3758,3851.0,7609.0,74.619605,101.970521
3,King,Federal Way,2737,604.0,3341.0,36.813315,90.755206
4,King,Kent,6091,2153.0,8244.0,48.740542,169.140508
5,King,Kirkland Totem Lake,15575,8227.0,23802.0,120.445340,197.616612
6,King,Redmond Downtown,11142,9678.0,20820.0,100.648714,206.858082
7,King,Redmond-Overlake,60566,7002.0,67568.0,204.199541,330.892027
8,King,Renton,16601,5480.0,22081.0,125.430441,176.041795
9,King,SeaTac,26970,11483.0,38453.0,99.633907,385.942909


# Notes

- data source: urbansim 2023_parcel_baseyear data
- Activity units = persons (from households table) + jobs
- home-based jobs are not included
- FAR is calculated as gross_sqft / land_area using the buildings table
- vacant land is not included